In [7]:
import src.grid_world.env as env
import numpy as np

size = 5
rng = np.random.default_rng(0)

human_1 = []
for i in range(size):
    human_1.append(lambda x, i=i: float(x.box[0] <= i))

human_2 = []
for i in range(size):
    human_2.append(lambda x, i=i: float(x.box[0] >= i))

population = [human_1, human_2]

func_env = env.GridWorldFuncEnv(size, population)
start_state = func_env.initial(rng)

state = env.GridWorldState(
    agent=(1, 1),
    target=(2, 2),
    box=(0, 0),
)

print(func_env.terminal(state))
func_env.reward(state, 0, state)

False


[[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]]

In [8]:
import src.grid_world.solvino as solvino

params = solvino.EmpoParameter(
    gamma_r=1,
    beta_r=1,
    gamma_h=1,
    zeta=2,
    xi=1,
    eta=1,
)

solver = solvino.BackwardInductionSolver(func_env, params)

solver.solve(state)

KeyError: (0, 0, GridWorldState(agent=(2, 2), target=(2, 2), box=(0, 0)))

In [6]:
solver.robot_policy[state]

KeyError: GridWorldState(agent=(2, 2), target=(2, 2), box=(1, 1))

# Temp tests

In [ ]:
import numpy as np

agent = np.array([2, 1])

print(np.clip(agent + [2, 2], 0, 3))
print(agent)

[3 3]
[2 1]


# Phase 2 Equations

\begin{equation}
Q_r(s,a_r) \gets \mathbb{E}_g \mathbb{E}_{a_H \sim \pi_H(s,g)} \mathbb{E}_{s' \sim s,a} \gamma_r V_r(s')
\tag{4}
\end{equation}

\begin{equation}
\pi_r(s)(a_r) \propto (-Q_r(s,a_r))^{-\beta_r}
\tag{5}
\end{equation}

\begin{equation}
V_h^e(s,g_h) \gets \mathbb{E}_{a_r \sim \pi_r(s)} \mathbb{E}_{g_{-h}} \mathbb{E}_{a_H \sim \pi_H(s,g)} \mathbb{E}_{s' \sim s,a} [U_h(s',g_h) + \gamma_h V_h^e(s', g_h)]
\tag{6}
\end{equation}

\begin{equation}
X_h(s) \gets \sum_{g_h \in \mathcal{G}_h} V_h^e(s,g_h)^\zeta
\tag{7}
\end{equation}

\begin{equation}
U_r(s) \gets - (\sum_h X_h(s)^{-\xi})^\eta
\tag{8}
\end{equation}
$U_r < 0$ and the closer to zero the better

\begin{equation}
V_r(s) \gets U_r(s) + \mathbb{E}_{a_r \sim \pi_r(s)} Q_r(s,a_r)
\tag{9}
\end{equation}

## Easy: One Human, One Goal, No Human Poliy, Deterministic Environment

\begin{equation}
Q_r(s,a) \gets  \gamma_r V_r(t(s,a))
\tag{4e}
\end{equation}

\begin{equation}
\pi(s) = \arg\max_a Q_r(s,a_r)
\tag{5e}
\end{equation}
or $\arg\min$?

\begin{equation}
V_h(s) \gets U_h(t(s,\pi(s))) + \gamma_h V_h(t(s,\pi(s)))
\tag{6e}
\end{equation}

\begin{equation}
X(s) \gets V_h(s)^\zeta
\tag{7e}
\end{equation}

\begin{equation}
U_r(s) \gets - (X(s)^{-\xi})^\eta
\tag{8e}
\end{equation}

or
$$
U_r(s) \gets - V_h(s)^{-\beta}
$$


\begin{equation}
V_r(s) \gets U_r(s) + Q_r\Big(s,\pi(s)\Big)
\tag{9e}
\end{equation}

So in the end
$$
V_r(s) \gets U_r(s) + \max_a \gamma V_r(t(s,a))
$$

## Add constant to $U_h$

Doesn't seem to just scale or add up...

If $U_h' = U_h + C$, then

$$
V_h' = V_h + C
$$

$$
X_h' = \sum (V_h')^\zeta = \sum (V_h + C)^\zeta
$$

$\zeta = 2$, $\eta = \xi = 1$

$$
X_h' = \sum (V_h + C)^2 = \sum V_h^2 + 2C \sum V_h + C^2|G| = X_h + C ( \sum 2V_h + C) = X_h + C(h)
$$

$$
U_r' = - \sum_h \frac{1}{X_h + C(h)}

## Special Case for goal independent human policy

If we assume $\pi_H(s,g) = \pi_H(s)$ then (4) and (6) simplify.

\begin{equation}
Q_r(s,a_r) \gets \mathbb{E}_{a_H \sim \pi_H(s)} \mathbb{E}_{s' \sim s,a} \gamma_r V_r(s')
\tag{4*}
\end{equation}

\begin{equation}
V_h^e(s,g_h) \gets \mathbb{E}_{a_r \sim \pi_r(s)} \mathbb{E}_{a_H \sim \pi_H(s)} \mathbb{E}_{s' \sim s,a} [U_h(s',g_h) + \gamma_h V_h^e(s', g_h)]
\tag{6*}
\end{equation}